In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def dustywave_roots(eps_list, ts_list, omega_s=2*np.pi):
    ndust = len(ts_list)

    #lambda = (1 - omega * ts) for each ts
    #so the coefficients of the polynomial are [-ts, 1.0]
    lambas = [np.poly1d([-ts, 1.0]) for ts in ts_list]
    
    #initialize the polynomial to 1
    lambda_poly = np.poly1d([1.0])

    #multiply the polynomials together
    for l in lambas:
        lambda_poly *= l
    
    #summation term
    summation = np.poly1d([0.0])

    for j in range(ndust):
        #calculate the product of all lambdas except for the j-th one
        product = np.poly1d([1.0])
        for k in range(ndust):
            if k != j:
                product *= lambas[k]
        
        #add the term to the summation
        #this is 1 + [(eps / lambda) for each dust species]
        summation += eps_list[j] * product 
    
    #omega^2 term
    omega2 = np.poly1d([1.0, 0.0, 0.0])  # Represents omega^2

    lhs = omega2 * (lambda_poly + summation)
    rhs = - omega_s**2 * lambda_poly


    #combine lhs and rhs to form the polynomial equation
    poly = lhs - rhs
    roots = np.roots(poly.coeffs)
    return roots


In [3]:
ts_list = [0.001]
eps_list = [2.24]

omegas = dustywave_roots(eps_list, ts_list)
print(omegas)

[3.23999158e+03+0.j        4.21200001e-03+3.4906605j
 4.21200001e-03-3.4906605j]


In [4]:
ts  = 0.001
eps = 2.24

omega_s = 2 * np.pi
# directly from (A6)
coeffs = [ts, -(1+eps), omega_s**2 * ts, -omega_s**2]
roots_cubic = np.roots(coeffs)
omega = roots_cubic.real + abs(roots_cubic.imag)*1j

print(omega)

[3.23999158e+03+0.j        4.21200001e-03+3.4906605j
 4.21200001e-03+3.4906605j]


In [5]:
def vg(omega, omega_s=2*np.pi): 
    return -1j * omega / omega_s

def vd_j(omega, ts_j, omega_s=2*np.pi): 
    return -1j * omega / omega_s * 1/(1 - omega * ts_j)

def rhod_j(omega, ts_j, eps_j): 
    return eps_j / (1 - omega * ts_j)

In [6]:
def strs_ans_multi(omega, ts_list, eps_list, amp=1e-4):
    result = [amp, 1.0, 0.0, vg(omega).real, vg(omega).imag]
    for ts, eps in zip(ts_list, eps_list):
        result += [rhod_j(omega, ts, eps).real, rhod_j(omega, ts, eps).imag, vd_j(omega, ts).real, vd_j(omega, ts).imag]
    return result

In [8]:
ts_list = [0.001, 1.0, 10] 
eps_list = np.ones(len(ts_list)) * 2.24 / len(ts_list)


#from Pablo's paper, this is checked to be correct
#eps_list = [0.1, 0.233333, 0.366667, 0.5]
#ts_list = [0.1, 0.215443, 0.464159, 1.0]

omega = dustywave_roots(eps_list, ts_list)
omega_wave = omega[(omega.imag < 0)][0]
omega_wave


(0.2303942807213155-4.703540235116413j)

In [22]:
strs = ['AMP', 'RERHOG', 'IMRHOG', 'REVX1G', 'IMVX1G']
for i in range(len(ts_list)):
    strs += [f'RERHOD{i+1}', f'IMRHOD{i+1}', f'REVX1D{i+1}', f'IMVX1D{i+1}']

In [31]:
ans = strs_ans_multi(omega_wave, ts_list, eps_list)

print(f'================================')
print('Stokes numbers:\n'+str(ts_list))
print('Dust-to-gas ratios:\n'+str(eps_list))
print(f'================================')

for key, value in zip(strs, ans): 
    print(f'{key}\t{value}')

Stokes numbers:
[0.001, 1.0, 10]
Dust-to-gas ratios:
[0.74666667 0.74666667 0.74666667]
AMP	0.0001
RERHOG	1.0
IMRHOG	0.0
REVX1G	-0.7485916784503928
IMVX1G	-0.03666838863689913
RERHOD1	0.7468222042589958
IMRHOD1	-0.0035135177806123814
REVX1D1	-0.7489201639630256
IMVX1D1	-0.033153450878277584
RERHOD2	0.025297123969429382
IMRHOD2	-0.1546064919247988
REVX1D2	-0.03295498315704768
IMVX1D2	0.1537627614056322
RERHOD3	-0.0004397460373613455
IMRHOD3	-0.015862376543820146
REVX1D3	-0.0003381128084107917
IMVX1D3	0.01592486766930799
